# 06 - Annotate and inspect results

Merges a results table with its LocusZoom annotation (rsIDs and allele
frequencies), writes the combined file, and loads it into GWASlab to pull lead
variants and draw Manhattan / QQ plots.

Works for both the METAL meta-analysis and a single-ancestry GWAS; pick the
section you need.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import gwaslab as gl

import gwas_config as cfg

pd.options.mode.chained_assignment = None

cfg.describe()

In [ ]:
# Mount the release and working buckets with gcsfuse (run once per VM).
for cmd in cfg.mount_commands():
    print(cmd)

## Meta-analysis: merge METAL with the LocusZoom annotation

The LocusZoom export is keyed on `#chrom`, `pos`, `ref`, `alt`; METAL's
`MarkerName` carries all four, so it is split out to join on.

In [ ]:
ANALYSIS = 'SURV'            # 'SURV' or 'CS_disc'
LABEL = f'HY_{ANALYSIS}_META'   # METAL prefix used in notebook 05

metal_dir = cfg.prep_path('meta', 'METAL', ANALYSIS)
lz_dir = metal_dir / 'locuszoom'

locuszoom = pd.read_csv(lz_dir / f'METAL_{ANALYSIS}.gz', sep='\t', compression='gzip')
display(locuszoom.head())

In [ ]:
metal = pd.read_csv(metal_dir / f'{LABEL}.TBL', sep='\t')

ids = metal['MarkerName'].str.split(':')
metal['#chrom'] = ids.str[0].str.replace('chr', '').astype(int)
metal['pos'] = ids.str[1].astype(int)
metal['ref'] = ids.str[2].str.upper()
metal['alt'] = ids.str[3].str.upper()

display(metal.head())

In [ ]:
combined = locuszoom.merge(metal, on=['#chrom', 'pos', 'ref', 'alt'], how='inner')
print(f'{len(combined)} of {len(metal)} METAL variants carry an annotation')

save_path = lz_dir / 'combined' / f'METAL_{ANALYSIS}.tsv.gz'
save_path.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(save_path, sep='\t', index=False, compression='gzip')
print(f'wrote {save_path}')

## Load into GWASlab

`Pvalue` is the fixed-effect p-value; switch to `PvalueARE` (and respective Beta and SE) for
random-effects.

In [ ]:
sumstats = gl.Sumstats(
    str(save_path),
    rsid='rsid',
    chrom='#chrom',
    pos='pos',
    ea='alt',
    nea='ref',
    eaf='alt_allele_freq',
    p='Pvalue',
    n='TotalSampleSize',
    beta='beta',
    se='stderr_beta',
    ref='ref',
    alt='alt',
    sep='\t',
)

In [ ]:
sumstats.get_lead(sig_level=5e-8, build='38')

In [ ]:
sumstats.plot_mqq()

In [ ]:
sumstats.plot_mqq(mode='qq', stratified=True)

In [ ]:
# Look up specific variants.
TOP_VARS = ['rs55833024']
df = sumstats.data
display(df[df.rsID.isin(TOP_VARS)])

## Single ancestry

Same merge against the pipeline's own GWAS output rather than METAL's.

In [ ]:
ANCESTRY = 'AFR'
ANALYSIS = 'CS_disc'

anc_dir = cfg.prep_path(*cfg.prep_parts(ancestry=ANCESTRY), ANALYSIS)
lz_dir = anc_dir / 'locuszoom'

locuszoom = pd.read_csv(lz_dir / f'{ANCESTRY}_{ANALYSIS}.gz', sep='\t',
                        compression='gzip')

gwas_path = (anc_dir / f'{ANCESTRY}_{ANALYSIS}_CHROMS_1_22' / 'gwas_results'
             / cfg.MODEL_DIRS[ANALYSIS]
             / f'{ANCESTRY}_case_control_y_allresults.tsv')
gwas = pd.read_csv(gwas_path, sep='\t')

gwas['#chrom'] = gwas['#CHROM'].str.replace('chr', '').astype(int)
gwas['pos'] = gwas['POS'].astype(int)
gwas['ref'] = gwas['REF']
gwas['alt'] = gwas['ALT']

combined = locuszoom.merge(gwas, on=['#chrom', 'pos', 'ref', 'alt'], how='inner')
combined = combined.drop(columns=['#CHROM', 'POS', 'REF', 'ALT'])
print(f'{len(combined)} annotated variants')

save_path = lz_dir / f'{ANCESTRY}_{ANALYSIS}.tsv.gz'
combined.to_csv(save_path, sep='\t', index=False, compression='gzip')
print(f'wrote {save_path}')

In [ ]:
sumstats = gl.Sumstats(
    str(save_path),
    rsid='rsid',
    chrom='#chrom',
    pos='pos',
    ea='alt',
    nea='ref',
    eaf='alt_allele_freq',
    p='P',
    n='OBS_CT',
    beta='BETA',
    se='SE',
    ref='ref',
    alt='alt',
    sep='\t',
)

sumstats.get_lead(sig_level=5e-8, build='38')